---
title: Weak Phase Notebook
authors: [gvarnavides]
date: 2026-04-08
---

In [ ]:
# imports
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

from IPython.display import display
import ipywidgets
import py4DSTEM

plt.rcParams['text.color']='white'
plt.rcParams['xtick.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.labelcolor'] = 'white'
plt.rcParams['ytick.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['axes.edgecolor'] = 'white'

width = 595
dpi = 72

In [2]:
# parameters
n = 192
k_max = 2 # inverse Angstroms
k_probe = 1 # inverse Angstroms
wavelength = 0.019687 # 300kV
sampling = 1 / k_max / 2 # Angstroms
reciprocal_sampling = 2 * k_max / n # inverse Angstroms
C10 = 2e2

In [3]:
# we build probe in Fourier space, using a soft aperture

kx = ky = np.fft.fftfreq(n,sampling)
k2 = kx[:,None]**2 + ky[None,:]**2
k  = np.sqrt(k2)
phi = np.arctan2(ky[None,:],kx[:,None])

unrolled_chi = np.pi*wavelength*C10*k2
def soft_aperture(k,k_probe=k_probe,reciprocal_sampling=reciprocal_sampling):
    """ """
    return np.sqrt(
        np.clip(
            (k_probe - k)/reciprocal_sampling + 0.5,
            0,
            1,
        ),
    )

def return_complex_probe(
    k,
    C10,
    wavelength=wavelength,
):
    """ """
    probe_array_fourier_0 = soft_aperture(k)
    probe_array_fourier_0 /= np.sqrt(np.sum(np.abs(probe_array_fourier_0)**2))
    return probe_array_fourier_0 * np.exp(-1j*np.pi*wavelength*C10*k**2)

aperture_fourier = soft_aperture(k)
probe_array_fourier = return_complex_probe(k,C10)

probe_array_fourier_centered = np.fft.fftshift(probe_array_fourier)
probe_array_rgb = py4DSTEM.visualize.Complex2RGB(probe_array_fourier_centered,vmin=0,vmax=1)

In [4]:
def return_aperture_overlap_factor(
    unrolled_chi,
    k_qm,
    k_qp
):
    """ """
    aperture_minus = soft_aperture(k_qm)
    aperture_plus = soft_aperture(k_qp)
    
    beta = np.exp(-1j*unrolled_chi)*aperture_minus - np.exp(1j*unrolled_chi)*aperture_plus
    beta_abs = np.abs(beta)
    beta_abs[beta_abs==0] = np.inf
    aperture_overlap_factor = beta.conj() / beta_abs
    return aperture_overlap_factor

def return_full_gamma_factor_k(
    complex_probe,
    k_qm,
    k_qp,
    C10,
):
    """ """
    shifted_probe_minus = return_complex_probe(k_qm,C10)
    shifted_probe_plus = return_complex_probe(k_qp,C10)
    gamma = shifted_probe_minus*complex_probe.conjugate() - shifted_probe_plus.conjugate()*complex_probe
    return gamma

def gamma_factor_at_k_index(
    kx_index,
    ky_index
):
    
    kx_qx = kx - kx[kx_index]
    ky_qy = ky - ky[ky_index]
    k_qm = np.sqrt(kx_qx[:,None]**2 + ky_qy[None,:]**2)
    
    kx_qx = kx + kx[kx_index]
    ky_qy = ky + ky[ky_index]
    k_qp = np.sqrt(kx_qx[:,None]**2 + ky_qy[None,:]**2)
    
    full_gamma_factor_k = return_full_gamma_factor_k(
        probe_array_fourier,
        k_qm,
        k_qp,
        C10
    )

    return full_gamma_factor_k

def gamma_factor_at_q_index(
    kx_index,
    ky_index
):
    
    kx_qx = kx - kx[kx_index]
    ky_qy = ky - ky[ky_index]
    k_qm = np.sqrt(kx_qx[:,None]**2 + ky_qy[None,:]**2)
    
    kx_qx = kx + kx[kx_index]
    ky_qy = ky + ky[ky_index]
    k_qp = np.sqrt(kx_qx[:,None]**2 + ky_qy[None,:]**2)
    
    complex_probe_factor = probe_array_fourier[kx_index,ky_index]
    full_gamma_factor_q = return_full_gamma_factor_k(
        complex_probe_factor,
        k_qm,
        k_qp,
        C10
    )

    return full_gamma_factor_q

def add_scalebar(ax, length, sampling, units, color="white", size_vertical=1, pad=0.5):
    """ """
    if np.mod(sampling*length,1) == 0:
        if int(np.round(sampling*length)) == 1:
            label = f"{units}"
        else: 
            label = f"{sampling*length:.0f} {units}"
    else:
        label=f"{sampling*length:.1f} {units}"

    bar = AnchoredSizeBar(
        ax.transData,
        length,
        label,
        "lower right",
        pad=pad,
        color=color,
        frameon=False,
        label_top=True,
        size_vertical=size_vertical,
    )
    ax.add_artist(bar)
    return ax, bar

In [5]:
gamma_q = np.fft.fftshift(gamma_factor_at_q_index(0,0))
gamma_q_rgb = py4DSTEM.visualize.Complex2RGB(gamma_q,vmin=0,vmax=1)

gamma_k = np.fft.fftshift(gamma_factor_at_k_index(0,0))
gamma_k_rgb = py4DSTEM.visualize.Complex2RGB(gamma_k,vmin=0,vmax=1)

In [ ]:
# visualization
aspect_ratio = 0.35
height = int(width * aspect_ratio)
with plt.ioff():
    fig,axs = plt.subplots(1,3, figsize=(width/dpi,height/dpi),dpi=dpi,layout="constrained")

im_k = axs[0].imshow(gamma_k_rgb)
axs[1].imshow(probe_array_rgb)
im_q = axs[2].imshow(gamma_q_rgb)

scatter_k = axs[0].scatter(n//2,n//2,color=(1, 47/85, 161/255))
scatter_q = axs[2].scatter(n//2,n//2,color='white')
scatter_k2 = axs[1].scatter(n//2,n//2,color=(1, 47/85, 161/255))

circle_k = patches.Circle(
    (n//2, n//2),
    radius=n//4,
    edgecolor='white',
    facecolor='none',
    ls='--',
    lw=1, alpha=0.8
)

circle_kpq = patches.Circle(
    (n//2, n//2),
    radius=n//4,
    edgecolor='white',
    facecolor='none',
    ls='--',
    lw=1, alpha=0.8
)

circle_kmq = patches.Circle(
    (n//2, n//2),
    radius=n//4,
    edgecolor='white',
    facecolor='none',
    ls='--',
    lw=1, alpha=0.8
)
for circle in [circle_k,circle_kmq,circle_kpq]:
    axs[0].add_patch(circle)


circle_qpk = patches.Circle(
    (n//2, n//2),
    radius=n//4,
    edgecolor='white',
    facecolor='none',
    ls='--',
    lw=1, alpha=0.8
)

circle_qmk = patches.Circle(
    (n//2, n//2),
    radius=n//4,
    edgecolor='white',
    facecolor='none',
    ls='--',
    lw=1, alpha=0.8
)
for circle in [circle_qpk,circle_qmk]:
    axs[2].add_patch(circle)

add_scalebar(
    axs[0],
    n//4,
    reciprocal_sampling,
    r"$k_{\mathrm{probe}}$",
    size_vertical=2.5,
)

add_scalebar(
    axs[1],
    n//4,
    reciprocal_sampling,
    r"$k_{\mathrm{probe}}$",
    size_vertical=2.5,
)

add_scalebar(
    axs[2],
    n//4,
    reciprocal_sampling,
    r"$q_{\mathrm{probe}}$",
    size_vertical=2.5,
)

for ax, title in zip(axs,[r"$\Gamma(q,k)$ @ fixed $q$","converged Fourier probe",r"$\Gamma(q,k)$ @ fixed $k$"]):
    ax.set(xticks=[],yticks=[],title=title)
fig

fig.canvas.resizable = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False
fig.canvas.toolbar_visible = False
fig.canvas.layout.width = f"{width}px"
# fig.canvas.layout.height = f"{height+10}px"
fig.patch.set_alpha(0)
None

In [9]:
#| label: app:gamma-function
slider_kv = ipywidgets.IntSlider(
    value=0, min=-n//2, max=n//2, step=1,
    readout=False, orientation='vertical',
    layout=ipywidgets.Layout(width="20px", height=f"{height-10}px")
)

slider_kh = ipywidgets.IntSlider(
    value=0, min=-n//2, max=n//2, step=1,
    readout=False, orientation='horizontal',
    layout=ipywidgets.Layout(width=f"{width//3}px", height="20px")
)

slider_qv = ipywidgets.IntSlider(
    value=0, min=-n//2, max=n//2, step=1,
    readout=False, orientation='vertical',
    layout=ipywidgets.Layout(width="20px", height=f"{height-10}px")
)

slider_qh = ipywidgets.IntSlider(
    value=0, min=-n//2, max=n//2, step=1,
    readout=False, orientation='horizontal',
    layout=ipywidgets.Layout(width=f"{width//3}px", height="20px")
)

def update_q_plot(change=None):
    qy, qx = slider_qv.value, slider_qh.value
    ky, kx = slider_kv.value, slider_kh.value

    gamma_q = np.fft.fftshift(gamma_factor_at_q_index(ky,kx))
    gamma_q_rgb = py4DSTEM.visualize.Complex2RGB(gamma_q, vmin=0, vmax=1)

    scatter_k.set_offsets([kx + n//2,-ky + n//2])
    scatter_k2.set_offsets([kx + n//2,-ky + n//2])
    circle_qpk.set_center((-kx + n//2,-ky + n//2))
    circle_qmk.set_center((kx + n//2,ky + n//2))
    im_q.set_data(gamma_q_rgb)
    label.value = f"q=({qx*reciprocal_sampling:.2f},{qy*reciprocal_sampling:.2f}), k=({kx*reciprocal_sampling:.2f},{ky*reciprocal_sampling:.2f})"
    fig.canvas.draw_idle()


def update_k_plot(change=None):
    qy, qx = slider_qv.value, slider_qh.value
    ky, kx = slider_kv.value, slider_kh.value
    
    gamma_k = np.fft.fftshift(gamma_factor_at_k_index(qy, qx))
    gamma_k_rgb = py4DSTEM.visualize.Complex2RGB(gamma_k, vmin=0, vmax=1)

    scatter_q.set_offsets([qx + n//2,-qy + n//2])
    circle_kpq.set_center((-qx + n//2,-qy + n//2))
    circle_kmq.set_center((qx + n//2,qy + n//2))
    
    im_k.set_data(gamma_k_rgb)
    label.value = f"q=({qx*reciprocal_sampling:.2f},{qy*reciprocal_sampling:.2f}), k=({kx*reciprocal_sampling:.2f},{ky*reciprocal_sampling:.2f})"
    fig.canvas.draw_idle()

slider_qv.observe(update_k_plot, names='value')
slider_qh.observe(update_k_plot, names='value')

slider_kv.observe(update_q_plot, names='value')
slider_kh.observe(update_q_plot, names='value')

buffer = ipywidgets.Label(
    layout=ipywidgets.Layout(width="20px", height="10px")
)

label = ipywidgets.HTMLMath(
    value=f"q = ({slider_qh.value:.2f}, {slider_qv.value:.2f}), k = ({slider_kh.value:.2f}, {slider_kv.value:.2f})",
    layout=ipywidgets.Layout(width=f"{width//3}px", display='flex', justify_content='center')
)

plot_plus_h_slider = ipywidgets.VBox([
    fig.canvas,
    ipywidgets.HBox([slider_qh, label,slider_kh])
])

app_layout = ipywidgets.HBox([ipywidgets.VBox([buffer,slider_qv]), plot_plus_h_slider,ipywidgets.VBox([buffer,slider_kv])])

app_layout